In [15]:

from pathlib import Path
import pandas as pd
import numpy as np

In [16]:
# Setting project paths

cwd = Path.cwd().resolve()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd

RAW_SYNTHETIC_PATH = (
    BASE_DIR / "data" / "synthetic" / "fleet_rotary_blower_Finaldataset_2025.csv"
)

PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs" / "modeling_data_prep"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_SYNTHETIC_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {RAW_SYNTHETIC_PATH}")

print("Base directory:", BASE_DIR)
print("Input file:", RAW_SYNTHETIC_PATH)

Base directory: C:\Users\USER\Documents\Shool projects\Predictive_Maintenance_Thesis
Input file: C:\Users\USER\Documents\Shool projects\Predictive_Maintenance_Thesis\data\synthetic\fleet_rotary_blower_Finaldataset_2025.csv


In [17]:
#  Loading synthetic fleet dataset

fleet_df = pd.read_csv(RAW_SYNTHETIC_PATH)

fleet_df["date"] = pd.to_datetime(fleet_df["date"])

fleet_df = fleet_df.sort_values(
    ["blower_id", "date"]
).copy()

print("Dataset loaded successfully.")
print("Shape:", fleet_df.shape)
print("Date range:", fleet_df["date"].min(), "to", fleet_df["date"].max())
print("Number of blowers:", fleet_df["blower_id"].nunique())

fleet_df.head()

Dataset loaded successfully.
Shape: (4380, 28)
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00
Number of blowers: 12


,date,amb_temp_max_c,amb_temp_min_c,amb_temp_avg_c,humidity,wind_gust_kph,blower_id,site_id,operational_class,max_op_ambient_temp_c,...,casing_temperature_c,vibration_mm_s,daily_degradation,degradation_index,maintenance_event,maintenance_threshold,health_score,maintenance_event_triggered,failure_event,RUL_days
0,2025-01-01,35.777778,23.555556,29.333333,63.9,17.2,ZG150_B001,Site_01,High Duty,48.9,...,50.359494,5.715649,0.007377,0.007377,0,0.6,99.262303,0,0,165.103543
1,2025-01-02,31.777778,20.722222,26.055556,79.5,15.0,ZG150_B001,Site_01,High Duty,48.9,...,41.829335,5.213525,0.005593,0.012970,0,0.6,98.703011,0,0,164.173270
2,2025-01-03,31.611111,22.222222,26.611111,78.3,13.9,ZG150_B001,Site_01,High Duty,48.9,...,41.794693,5.345015,0.005264,0.018233,0,0.6,98.176655,0,0,163.297779
3,2025-01-04,32.722222,23.722222,28.000000,71.4,12.9,ZG150_B001,Site_01,High Duty,48.9,...,46.457210,4.884268,0.005700,0.023934,0,0.6,97.606606,0,0,162.349612
4,2025-01-05,34.111111,22.500000,29.388889,66.3,19.8,ZG150_B001,Site_01,High Duty,48.9,...,49.750704,5.307526,0.006669,0.030603,0,0.6,96.939732,0,0,161.240397


In [18]:
# Basic data quality checks

print("Missing values:")
display(fleet_df.isna().sum())

print("\nDuplicate rows:")
print(fleet_df.duplicated().sum())

print("\nMaintenance event distribution:")
display(fleet_df["maintenance_event"].value_counts())

print("\nFailure event distribution:")
display(fleet_df["failure_event"].value_counts())

Missing values:


date                           0
amb_temp_max_c                 0
amb_temp_min_c                 0
amb_temp_avg_c                 0
humidity                       0
wind_gust_kph                  0
blower_id                      0
site_id                        0
operational_class              0
max_op_ambient_temp_c          0
failure_threshold              0
max_op_rpm                     0
daily_op_hours                 0
cumulative_op_hours            0
daily_load_percent             0
rpm                            0
dust_index                     0
pressure_diff_psi              0
casing_temperature_c           0
vibration_mm_s                 0
daily_degradation              0
degradation_index              0
maintenance_event              0
maintenance_threshold          0
health_score                   0
maintenance_event_triggered    0
failure_event                  0
RUL_days                       0
dtype: int64


Duplicate rows:
0

Maintenance event distribution:


maintenance_event
0    4343
1      37
Name: count, dtype: int64


Failure event distribution:


failure_event
0    4380
Name: count, dtype: int64

In [19]:
# Checking required columns

required_columns = [
    "date",
    "blower_id",
    "site_id",
    "operational_class",
    "daily_op_hours",
    "cumulative_op_hours",
    "daily_load_percent",
    "humidity",
    "wind_gust_kph",
    "dust_index",
    "pressure_diff_psi",
    "casing_temperature_c",
    "vibration_mm_s",
    "daily_degradation",
    "degradation_index",
    "health_score",
    "maintenance_event",
    "failure_event",
    "RUL_days"
]

missing_required_columns = [
    col for col in required_columns if col not in fleet_df.columns
]

if missing_required_columns:
    raise ValueError(
        f"The following required columns are missing: {missing_required_columns}"
    )

print("All required columns are available.")

All required columns are available.


In [20]:
# Creating future maintenance target variables. These indicate whether maintenance occurs within the next 90 days.

fleet_df["maintenance_due_90d"] = (
    fleet_df.groupby("blower_id")["maintenance_event"]
    .transform(lambda x: x[::-1].rolling(window=90, min_periods=1).max()[::-1])
    .astype(int)
)

fleet_df[[
    "date",
    "blower_id",
    "maintenance_event",
    "maintenance_due_90d"
]].head(20)

,date,blower_id,maintenance_event,maintenance_due_90d
0,2025-01-01,ZG150_B001,0,0
1,2025-01-02,ZG150_B001,0,0
2,2025-01-03,ZG150_B001,0,0
3,2025-01-04,ZG150_B001,0,0
4,2025-01-05,ZG150_B001,0,0
5,2025-01-06,ZG150_B001,0,0
6,2025-01-07,ZG150_B001,0,0
7,2025-01-08,ZG150_B001,0,1
8,2025-01-09,ZG150_B001,0,1
9,2025-01-10,ZG150_B001,0,1


In [21]:
# Creating maintenance history features

fleet_df = fleet_df.sort_values(
    ["blower_id", "date"]
).copy()

fleet_df["previous_maintenance_event"] = (
    fleet_df.groupby("blower_id")["maintenance_event"]
    .shift(1)
    .fillna(0)
    .astype(int)
)

fleet_df["maintenance_cycle"] = (
    fleet_df.groupby("blower_id")["previous_maintenance_event"]
    .cumsum()
    .astype(int)
)

fleet_df["days_since_maintenance"] = (
    fleet_df.groupby(["blower_id", "maintenance_cycle"])
    .cumcount()
)

fleet_df["cycle_start_cumulative_hours"] = (
    fleet_df.groupby(["blower_id", "maintenance_cycle"])["cumulative_op_hours"]
    .transform("first")
)

fleet_df["hours_since_maintenance"] = (
    fleet_df["cumulative_op_hours"] - fleet_df["cycle_start_cumulative_hours"]
)

fleet_df[[
    "date",
    "blower_id",
    "maintenance_event",
    "previous_maintenance_event",
    "maintenance_cycle",
    "days_since_maintenance",
    "hours_since_maintenance"
]].head(20)

,date,blower_id,maintenance_event,previous_maintenance_event,maintenance_cycle,days_since_maintenance,hours_since_maintenance
0,2025-01-01,ZG150_B001,0,0,0,0,0.000000
1,2025-01-02,ZG150_B001,0,0,0,1,20.440024
2,2025-01-03,ZG150_B001,0,0,0,2,43.565701
3,2025-01-04,ZG150_B001,0,0,0,3,66.976548
4,2025-01-05,ZG150_B001,0,0,0,4,86.976548
5,2025-01-06,ZG150_B001,0,0,0,5,107.023278
6,2025-01-07,ZG150_B001,0,0,0,6,129.215039
7,2025-01-08,ZG150_B001,0,0,0,7,150.740675
8,2025-01-09,ZG150_B001,0,0,0,8,172.715473
9,2025-01-10,ZG150_B001,0,0,0,9,193.435908


In [22]:
# Defining modelling features and targets

model_features = [
    "daily_op_hours",
    "cumulative_op_hours",
    "days_since_maintenance",
    "hours_since_maintenance",
    "daily_load_percent",
    "humidity",
    "wind_gust_kph",
    "dust_index",
    "pressure_diff_psi",
    "casing_temperature_c",
    "vibration_mm_s",
    "operational_class",
]

regression_target = "RUL_days"

classification_targets = [
    "maintenance_due_90d"
]

print("Regression target:", regression_target)
print("Classification targets:", classification_targets)
print("\nModel features:")
for feature in model_features:
    print("-", feature)

Regression target: RUL_days
Classification targets: ['maintenance_due_90d']

Model features:
- daily_op_hours
- cumulative_op_hours
- days_since_maintenance
- hours_since_maintenance
- daily_load_percent
- humidity
- wind_gust_kph
- dust_index
- pressure_diff_psi
- casing_temperature_c
- vibration_mm_s
- operational_class


In [23]:
# Checking missing values in modelling features and targets

modeling_columns = model_features + [regression_target] + classification_targets

missing_modeling_values = fleet_df[modeling_columns].isna().sum()

print("Missing values in modelling columns:")
display(missing_modeling_values[missing_modeling_values > 0])

if missing_modeling_values.sum() == 0:
    print("No missing values found in modelling columns.")

Missing values in modelling columns:


Series([], dtype: int64)

No missing values found in modelling columns.


In [24]:
# Checking target distributions

print("RUL target summary:")
display(fleet_df["RUL_days"].describe())

print("\nMaintenance due within 90 days:")
display(fleet_df["maintenance_due_90d"].value_counts())

RUL target summary:


count    4380.000000
mean      129.184593
std        32.852497
min        47.878817
25%       103.366261
50%       130.196410
75%       154.799169
max       212.804780
Name: RUL_days, dtype: float64


Maintenance due within 90 days:


maintenance_due_90d
1    3299
0    1081
Name: count, dtype: int64

In [25]:
# Saving feature and target definitions for consistency across RF and XGBoost notebooks

feature_definition = pd.DataFrame({
    "column_name": model_features + [regression_target] + classification_targets,
    "role": (
        ["model_feature"] * len(model_features)
        + ["regression_target"]
        + ["classification_target"] * len(classification_targets)
    )
})

feature_definition_path = OUTPUT_DIR / "model_feature_target_definition.csv"

feature_definition.to_csv(
    feature_definition_path,
    index=False
)

feature_definition

,column_name,role
0,daily_op_hours,model_feature
1,cumulative_op_hours,model_feature
2,days_since_maintenance,model_feature
3,hours_since_maintenance,model_feature
4,daily_load_percent,model_feature
5,humidity,model_feature
6,wind_gust_kph,model_feature
7,dust_index,model_feature
8,pressure_diff_psi,model_feature
9,casing_temperature_c,model_feature


In [26]:
# createing a compact modelling data dictionary for newly engineered fields

new_feature_dictionary = pd.DataFrame([
    {
        "column_name": "maintenance_due_90d",
        "description": "Binary target indicating whether a maintenance event occurs within the next 90 days",
        "unit": "0/1",
        "data_type": "integer",
        "source": "Derived from maintenance_event",
        "category": "Classification target"
    },
    {
        "column_name": "previous_maintenance_event",
        "description": "Binary indicator showing whether maintenance occurred on the previous day",
        "unit": "0/1",
        "data_type": "integer",
        "source": "Derived from maintenance_event",
        "category": "Maintenance history feature"
    },
    {
        "column_name": "maintenance_cycle",
        "description": "Cumulative maintenance cycle number for each blower",
        "unit": "cycle count",
        "data_type": "integer",
        "source": "Derived from previous_maintenance_event",
        "category": "Maintenance history feature"
    },
    {
        "column_name": "days_since_maintenance",
        "description": "Number of days elapsed since the last maintenance event",
        "unit": "days",
        "data_type": "integer",
        "source": "Derived from maintenance_cycle and date sequence",
        "category": "Maintenance history feature"
    },
    {
        "column_name": "cycle_start_cumulative_hours",
        "description": "Cumulative operating hours at the start of the current maintenance cycle",
        "unit": "hours",
        "data_type": "float",
        "source": "Derived from cumulative_op_hours and maintenance_cycle",
        "category": "Intermediate maintenance feature"
    },
    {
        "column_name": "hours_since_maintenance",
        "description": "Operating hours elapsed since the last maintenance event",
        "unit": "hours",
        "data_type": "float",
        "source": "Derived from cumulative_op_hours and cycle_start_cumulative_hours",
        "category": "Maintenance history feature"
    }
])

new_feature_dictionary_path = OUTPUT_DIR / "engineered_modeling_features_dictionary.csv"

new_feature_dictionary.to_csv(
    new_feature_dictionary_path,
    index=False
)

new_feature_dictionary

,column_name,description,unit,data_type,source,category
0,maintenance_due_90d,Binary target indicating whether a maintenance...,0/1,integer,Derived from maintenance_event,Classification target
1,previous_maintenance_event,Binary indicator showing whether maintenance o...,0/1,integer,Derived from maintenance_event,Maintenance history feature
2,maintenance_cycle,Cumulative maintenance cycle number for each b...,cycle count,integer,Derived from previous_maintenance_event,Maintenance history feature
3,days_since_maintenance,Number of days elapsed since the last maintena...,days,integer,Derived from maintenance_cycle and date sequence,Maintenance history feature
4,cycle_start_cumulative_hours,Cumulative operating hours at the start of the...,hours,float,Derived from cumulative_op_hours and maintenan...,Intermediate maintenance feature
5,hours_since_maintenance,Operating hours elapsed since the last mainten...,hours,float,Derived from cumulative_op_hours and cycle_sta...,Maintenance history feature


In [27]:
# Saving final modelling dataset

modeling_dataset_path = PROCESSED_DIR / "modeling_dataset.csv"

fleet_df.to_csv(
    modeling_dataset_path,
    index=False
)

print("Modelling dataset saved successfully.")
print("Saved to:", modeling_dataset_path)
print("Final shape:", fleet_df.shape)

Modelling dataset saved successfully.
Saved to: C:\Users\USER\Documents\Shool projects\Predictive_Maintenance_Thesis\data\processed\modeling_dataset.csv
Final shape: (4380, 34)


In [28]:
# Final verification by reloading saved dataset

check_df = pd.read_csv(modeling_dataset_path)
check_df["date"] = pd.to_datetime(check_df["date"])

print("Reloaded saved modelling dataset.")
print("Shape:", check_df.shape)
print("Date range:", check_df["date"].min(), "to", check_df["date"].max())
print("Number of blowers:", check_df["blower_id"].nunique())

check_df.head()

Reloaded saved modelling dataset.
Shape: (4380, 34)
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00
Number of blowers: 12


,date,amb_temp_max_c,amb_temp_min_c,amb_temp_avg_c,humidity,wind_gust_kph,blower_id,site_id,operational_class,max_op_ambient_temp_c,...,health_score,maintenance_event_triggered,failure_event,RUL_days,maintenance_due_90d,previous_maintenance_event,maintenance_cycle,days_since_maintenance,cycle_start_cumulative_hours,hours_since_maintenance
0,2025-01-01,35.777778,23.555556,29.333333,63.9,17.2,ZG150_B001,Site_01,High Duty,48.9,...,99.262303,0,0,165.103543,0,0,0,0,22.457076,0.000000
1,2025-01-02,31.777778,20.722222,26.055556,79.5,15.0,ZG150_B001,Site_01,High Duty,48.9,...,98.703011,0,0,164.173270,0,0,0,1,22.457076,20.440024
2,2025-01-03,31.611111,22.222222,26.611111,78.3,13.9,ZG150_B001,Site_01,High Duty,48.9,...,98.176655,0,0,163.297779,0,0,0,2,22.457076,43.565701
3,2025-01-04,32.722222,23.722222,28.000000,71.4,12.9,ZG150_B001,Site_01,High Duty,48.9,...,97.606606,0,0,162.349612,0,0,0,3,22.457076,66.976548
4,2025-01-05,34.111111,22.500000,29.388889,66.3,19.8,ZG150_B001,Site_01,High Duty,48.9,...,96.939732,0,0,161.240397,0,0,0,4,22.457076,86.976548
